# LoRA × RoBERTa-base — Replicating GLUE Table 2

This notebook uses the `project_LoRA` repo's from-scratch LoRA implementation to fine-tune `roberta-base` on GLUE and reproduce the LoRA row of **Table 2** from [Hu et al., 2021](https://arxiv.org/abs/2106.09685).

**Target hyperparameters (paper Table 9, RoBERTa-base + LoRA):**
- Target modules: `W_q`, `W_v`
- Rank `r = 8`, scaling `α = 16`, dropout 0.1
- Optimizer: AdamW, weight decay 0.1, linear schedule with 6% warmup
- Trainable params: ~0.3M (LoRA adapters + randomly-initialised classifier head)

**Runtime budget (Colab T4, free tier):** the small tasks (MRPC, RTE, CoLA, STS-B) each fit in ~20–60 min. SST-2 and QNLI are several hours. MNLI and QQP won't complete in a single free-tier session — run them on a paid/local GPU with the same config.

In [ ]:
!nvidia-smi

## 1. Load the project code

Pick **one** of the two options below.

**Option A (recommended):** zip the `code/` folder from your repo and upload it here.
```bash
# on your laptop, from inside project_LoRA/:
zip -r code.zip code -x 'code/external/*' -x '*/__pycache__/*' -x '*/.venv/*'
```

**Option B:** clone the full repo from a public GitHub mirror (only if the repo is pushed somewhere Colab can reach without SSH keys).

In [ ]:
# --- Option A: upload code.zip ---
from google.colab import files
import os, zipfile

uploaded = files.upload()  # drag-drop code.zip (or a zip of the whole repo)
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content")

# auto-discover the code/ root (the directory containing requirements.txt + lora/),
# searching up to 2 levels deep so it works whether you zipped `code/` or the whole repo
def find_code_root(base="/content", max_depth=2):
    for dirpath, _, filenames in os.walk(base):
        depth = dirpath[len(base):].count(os.sep)
        if depth > max_depth:
            continue
        if "requirements.txt" in filenames and os.path.isdir(os.path.join(dirpath, "lora")):
            return dirpath
    return None

root = find_code_root()
assert root is not None, "Couldn't find a folder with requirements.txt + lora/ in the uploaded zip."
os.chdir(root)
print("Working dir:", os.getcwd())
print(os.listdir(root))

In [ ]:
# --- Option B: clone from a public GitHub mirror (uncomment if applicable) ---
!git clone https://github.com/austinlu2005/myLoRA.git /content/myLoRA
%cd /content/myLoRA

In [ ]:
!pip install -q -r requirements.txt

## 2. Imports

In [ ]:
import sys, os, json, math, time
sys.path.insert(0, os.getcwd())

import torch
from transformers import AutoTokenizer

from models.roberta_wrapper import build_roberta_lora
from dataloaders.glue import load_glue
from evaluation.glue_metrics import compute_glue_metrics
from training.trainer import Trainer
from training.optim import build_optimizer, build_scheduler
from utils.seed import set_seed
from utils.param_utils import print_trainable_parameters

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 3. Paper hyperparameters (Table 9)

Per-task settings, all using AdamW + linear warmup schedule + weight decay 0.1 + LoRA(r=8, α=16) on `W_q`, `W_v`.

| Task | LR | Batch | Epochs | Max len | Metric |
|------|----|-------|--------|---------|--------|
| MNLI  | 5e-4 | 16 | 30 | 128 | accuracy (matched) |
| SST-2 | 5e-4 | 16 | 60 | 128 | accuracy |
| MRPC  | 4e-4 | 16 | 30 | 512 | accuracy |
| CoLA  | 4e-4 | 16 | 80 | 128 | Matthews |
| QNLI  | 4e-4 | 16 | 25 | 128 | accuracy |
| QQP   | 5e-4 | 16 | 25 | 128 | accuracy |
| RTE   | 5e-4 | 16 | 80 | 512 | accuracy |
| STS-B | 4e-4 | 16 | 40 | 128 | Pearson |

In [ ]:
GLUE_HPARAMS = {
    "mnli": {"num_labels": 3, "lr": 5e-4, "batch_size": 16, "num_epochs": 30, "max_length": 128, "is_regression": False},
    "sst2": {"num_labels": 2, "lr": 5e-4, "batch_size": 16, "num_epochs": 60, "max_length": 128, "is_regression": False},
    "mrpc": {"num_labels": 2, "lr": 4e-4, "batch_size": 16, "num_epochs": 30, "max_length": 512, "is_regression": False},
    "cola": {"num_labels": 2, "lr": 4e-4, "batch_size": 16, "num_epochs": 80, "max_length": 128, "is_regression": False},
    "qnli": {"num_labels": 2, "lr": 4e-4, "batch_size": 16, "num_epochs": 25, "max_length": 128, "is_regression": False},
    "qqp":  {"num_labels": 2, "lr": 5e-4, "batch_size": 16, "num_epochs": 25, "max_length": 128, "is_regression": False},
    "rte":  {"num_labels": 2, "lr": 5e-4, "batch_size": 16, "num_epochs": 80, "max_length": 512, "is_regression": False},
    "stsb": {"num_labels": 1, "lr": 4e-4, "batch_size": 16, "num_epochs": 40, "max_length": 128, "is_regression": True},
}

COMMON = {
    "rank": 8, "alpha": 16, "dropout": 0.1,
    "weight_decay": 0.1, "warmup_ratio": 0.06,
    "seed": 42, "grad_clip": 1.0,
}

# LoRA (r=8) row of Table 2 in the paper
PAPER_RESULTS = {
    "mnli": 87.5, "sst2": 95.1, "mrpc": 89.7, "cola": 63.4,
    "qnli": 93.3, "qqp":  90.8, "rte":  86.6, "stsb": 91.5,
}

PRIMARY_METRIC = {
    "mnli": "accuracy", "sst2": "accuracy", "mrpc": "accuracy",
    "cola": "matthews_correlation", "qnli": "accuracy",
    "qqp": "accuracy", "rte": "accuracy", "stsb": "pearson",
}

## 4. Regression-aware Trainer

The base `Trainer` in `training/trainer.py` does `logits.argmax(dim=-1)` at eval time, which is correct for classification but breaks for STS-B (regression, `num_labels=1`). Subclass it here to squeeze the logits instead.

In [ ]:
class RegressionTrainer(Trainer):
    @torch.no_grad()
    def evaluate(self):
        self.model.eval()
        all_preds, all_labels = [], []
        total_loss, n_batches = 0.0, 0
        for batch in self.eval_loader:
            batch = self._move_to_device(batch)
            out = self.model(**batch)
            total_loss += out.loss.item(); n_batches += 1
            all_preds.append(out.logits.squeeze(-1).detach().cpu())
            all_labels.append(batch["labels"].detach().cpu())
        preds = torch.cat(all_preds); labels = torch.cat(all_labels)
        metrics = {"eval_loss": total_loss / max(n_batches, 1)}
        if self.compute_metrics is not None:
            metrics.update(self.compute_metrics(preds, labels))
        return metrics

## 5. Training function

Wraps the full pipeline for one task: build model → load data → train → save `result.json`.

In [ ]:
RUNS_DIR = "/content/runs"
os.makedirs(RUNS_DIR, exist_ok=True)

def train_glue(task, hparams, common=COMMON, runs_dir=RUNS_DIR, model_name="roberta-base"):
    set_seed(common["seed"])

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model, replaced = build_roberta_lora(
        model_name=model_name,
        num_labels=hparams["num_labels"],
        rank=common["rank"], alpha=common["alpha"], dropout=common["dropout"],
    )
    print(f"[{task}] injected LoRA into {len(replaced)} modules")
    print_trainable_parameters(model)

    data = load_glue(task, tokenizer, max_length=hparams["max_length"])
    train_ds = data["train"]
    eval_split = "validation_matched" if task == "mnli" else "validation"
    eval_ds = data[eval_split]
    print(f"[{task}] train={len(train_ds)}  eval={len(eval_ds)}")

    steps_per_epoch = math.ceil(len(train_ds) / hparams["batch_size"])
    num_training_steps = steps_per_epoch * hparams["num_epochs"]

    optimizer = build_optimizer(model, lr=hparams["lr"], weight_decay=common["weight_decay"])
    scheduler = build_scheduler(optimizer, num_training_steps, warmup_ratio=common["warmup_ratio"])

    device = "cuda" if torch.cuda.is_available() else "cpu"

    def metrics_fn(preds, labels):
        return compute_glue_metrics(task, preds.numpy(), labels.numpy())

    TrainerCls = RegressionTrainer if hparams["is_regression"] else Trainer
    out_dir = f"{runs_dir}/{task}"
    trainer = TrainerCls(
        model=model, train_dataset=train_ds, eval_dataset=eval_ds,
        optimizer=optimizer, scheduler=scheduler,
        batch_size=hparams["batch_size"], num_epochs=hparams["num_epochs"],
        device=device, compute_metrics=metrics_fn,
        log_steps=100, grad_clip=common["grad_clip"], output_dir=out_dir,
    )

    t0 = time.time()
    trainer.train()
    wall = time.time() - t0

    key = PRIMARY_METRIC[task]
    best = max((h[key] for h in trainer.history if key in h), default=None)
    result = {
        "task": task, "metric_name": key, "best_metric": best,
        "wall_seconds": wall, "history": trainer.history,
        "hparams": hparams, "common": common,
    }
    with open(f"{out_dir}/result.json", "w") as f:
        json.dump(result, f, indent=2, default=str)
    print(f"[{task}] best {key} = {best:.4f}  (paper {PAPER_RESULTS[task]/100:.4f})  | wall {wall/60:.1f} min")
    return result

## 6. Quick sanity run: MRPC

MRPC is small (3.7k train examples), so 30 epochs at max_len 512 on T4 takes roughly 30–45 minutes. Use this as an end-to-end smoke test before launching longer runs.

In [ ]:
result_mrpc = train_glue("mrpc", GLUE_HPARAMS["mrpc"])

## 7. Run the remaining feasible tasks

Rough wall-time estimates on a T4 at paper hyperparameters (single seed):

| Task | Train size | Epochs × MaxLen | T4 estimate |
|------|-----------|------------------|-------------|
| RTE   | 2.5k   | 80 × 512 | ~30–45 min |
| CoLA  | 8.5k   | 80 × 128 | ~30–45 min |
| STS-B | 5.7k   | 40 × 128 | ~15–25 min |
| MRPC  | 3.7k   | 30 × 512 | ~30–45 min |
| SST-2 | 67k    | 60 × 128 | ~3–4 h |
| QNLI  | 105k   | 25 × 128 | ~3–5 h |
| QQP   | 364k   | 25 × 128 | **>10 h** |
| MNLI  | 393k   | 30 × 128 | **>12 h** |

Colab free tier disconnects after ~12 hours (and idle-kicks after ~90 min), so MNLI/QQP need Colab Pro/A100 or a local GPU. Edit `TASKS` below to the subset you want to run.

In [ ]:
TASKS = ["rte", "cola", "stsb"]  # add "sst2", "qnli", "mnli", "qqp" as your budget allows

results = {}
for task in TASKS:
    print("\n" + "=" * 70)
    print(f"  {task.upper()}")
    print("=" * 70)
    try:
        results[task] = train_glue(task, GLUE_HPARAMS[task])
    except Exception as e:
        import traceback; traceback.print_exc()
        results[task] = {"task": task, "error": str(e)}

## 8. Results table (LoRA paper, Table 2 format)

In [ ]:
import pandas as pd

TASK_ORDER = ["mnli", "sst2", "mrpc", "cola", "qnli", "qqp", "rte", "stsb"]

rows = []
for task in TASK_ORDER:
    path = f"{RUNS_DIR}/{task}/result.json"
    ours = None
    if os.path.isfile(path):
        with open(path) as f:
            r = json.load(f)
        if r.get("best_metric") is not None:
            ours = r["best_metric"] * 100
    paper = PAPER_RESULTS[task]
    delta = (ours - paper) if ours is not None else None
    rows.append({
        "task": task.upper(),
        "metric": PRIMARY_METRIC[task],
        "paper (LoRA r=8)": f"{paper:.1f}",
        "ours": f"{ours:.1f}" if ours is not None else "–",
        "Δ": f"{delta:+.1f}" if delta is not None else "–",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

## 9. (Optional) Training curves

In [ ]:
import matplotlib.pyplot as plt

completed = [t for t in TASK_ORDER if os.path.isfile(f"{RUNS_DIR}/{t}/result.json")]
if completed:
    fig, axes = plt.subplots(1, len(completed), figsize=(4*len(completed), 3), squeeze=False)
    for ax, t in zip(axes[0], completed):
        with open(f"{RUNS_DIR}/{t}/result.json") as f:
            hist = json.load(f)["history"]
        key = PRIMARY_METRIC[t]
        xs = [h["epoch"] for h in hist if key in h]
        ys = [h[key] for h in hist if key in h]
        ax.plot(xs, ys, marker="o"); ax.set_title(t.upper())
        ax.set_xlabel("epoch"); ax.set_ylabel(key); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("No completed runs yet.")

## 10. Notes & caveats

- **Single seed.** The paper reports the median of 5 seeds. To match properly, loop `common["seed"]` over `{0, 1, 2, 3, 4}` and take the median of `best_metric`. Multiplies wall time by 5.
- **MNLI / QQP on Colab free tier.** The datasets are ~400k examples. Even at 25–30 epochs they exceed the free-tier budget. Options: (a) reduce epochs to 5–10 and expect a ~1–2 point drop, (b) run on A100/V100, (c) run locally on a single 3090/4090.
- **Checkpoints.** `Trainer` saves only the LoRA adapter (`lora_best.pt`) and classifier head (`classifier_best.pt`) — ~1.5 MB per task, not the full model. Reload with `load_lora_state_dict`.
- **Colab disconnects.** Use `Runtime → Manage sessions` to keep the session alive. Results are on the ephemeral `/content` disk — copy `/content/runs` to Google Drive before disconnecting if you want to keep them.

```python
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/runs /content/drive/MyDrive/lora_runs
```